# 04 — MCP server from a Python client

The engine ships an MCP server (`engine.mcp.server`) that exposes three tools: `research`, `reset_memory`, `memory_count`. In production users configure this in Claude Desktop or Cursor. Here we spin it up in-process and call its tools directly — useful when you're building another agent that wants to invoke the engine.

**You will learn:**
1. What MCP is (briefly) and where the engine's tools live.
2. How to call the `research` tool without the full MCP wire protocol.
3. The exact shape of the JSON payload the tool returns.
4. How the Claude Desktop config references this server.

**Time:** ~3 minutes. **Cost:** $0. **Prereqs:** none.

## Step 1 — install

In [ ]:
%%capture
!git clone --depth 1 https://github.com/TheAiSingularity/agentic-research-engine-oss.git /content/engine-repo
!pip install -q langgraph openai rank_bm25 trafilatura pypdf mcp pyyaml

import sys, os
sys.path.insert(0, '/content/engine-repo')

## Step 2 — what the server exposes

`engine/mcp/server.py` defines three `@mcp.tool()` functions. You can inspect them without running the server.

In [ ]:
from engine.mcp import server as mcp_server

print('Server name:', mcp_server.mcp.name)
print()
print('Available tools (from the @mcp.tool() decorated functions):')
for fn_name in ('research', 'reset_memory', 'memory_count'):
    fn = getattr(mcp_server, fn_name)
    print(f'  - {fn_name}({fn.__doc__.splitlines()[0][:60]}...)' if fn.__doc__ else f'  - {fn_name}')

## Step 3 — call `research` directly

The `research` function is a regular Python callable; the `@mcp.tool()` decorator adds a JSON-RPC wrapper for the wire protocol but leaves the body callable. Perfect for testing.

We mock the LLM so this runs in seconds without API keys. In real usage, you'd route via `OPENAI_BASE_URL` as in Tutorial 02.

In [ ]:
# Mock LLM (same pattern as Tutorial 01).
from types import SimpleNamespace
from unittest import mock
import engine.core.models as _models
import engine.core.pipeline as _pipeline
import requests

def _resp(t): return SimpleNamespace(choices=[SimpleNamespace(message=SimpleNamespace(content=t))])
def canned(*a, **kw):
    p = kw.get('messages', [{}])[0].get('content', '')
    if 'Classify' in p:                      return _resp('factoid')
    if 'step-level verifier' in p:            return _resp('VERDICT: accept\nFEEDBACK:')
    if 'Break this research question' in p:   return _resp('what is MCP\nhow do servers work')
    if 'Summarize these sources' in p:         return _resp('MCP is a protocol [1]. Servers expose tools [2].')
    if 'Compress each numbered chunk' in p:   return _resp('[1] MCP is an open JSON-RPC protocol.\n[2] Servers expose tool schemas.')
    if 'Answer the question using ONLY' in p: return _resp('MCP (Model Context Protocol) is a JSON-RPC standard for agent tools [1][2].')
    if 'List each standalone factual' in p:   return _resp('CLAIM: MCP is JSON-RPC based\nVERIFIED: yes\nCLAIM: servers expose tool schemas\nVERIFIED: yes')
    return _resp('(stub)')

cli = mock.MagicMock(); cli.chat.completions.create.side_effect = canned
_models.OpenAI = mock.MagicMock(return_value=cli)

def fake_get(url, params=None, timeout=None):
    r = mock.MagicMock(); r.status_code = 200; r.raise_for_status = mock.MagicMock()
    r.json = lambda: {'results': [
        {'url': 'https://modelcontextprotocol.io', 'title': 'MCP spec', 'content': 'Model Context Protocol homepage'},
        {'url': 'https://github.com/modelcontextprotocol/servers', 'title': 'MCP servers', 'content': 'Reference implementations'},
    ]}
    return r
requests.get = fake_get

# Ensure hybrid retriever embedder is faked too.
from core.rag import HybridRetriever
orig_init = HybridRetriever.__init__
def patched(self, *a, **kw):
    orig_init(self, *a, **kw)
    self.embedder = lambda batch: [[float(len(s)), float(len(s.split()))] for s in batch]
HybridRetriever.__init__ = patched

_pipeline.ENABLE_STREAM = False
_pipeline.ENABLE_FETCH  = False

print('Mocks installed.')

In [ ]:
import json

# This is what an MCP client would see on the wire (just the tool's return value).
result = mcp_server.research(
    question='What is MCP and how do MCP servers work?',
    domain='general',
    memory='session',
)
print(json.dumps(result, indent=2, default=str))

## Step 4 — the other two tools

`reset_memory` and `memory_count` operate on the persistent SQLite store at `~/.agentic-research/memory.db`. In Colab this is ephemeral — the file dies when the runtime disconnects.

In [ ]:
print('memory_count:', mcp_server.memory_count())
# Uncomment to actually wipe the (ephemeral Colab) store:
# print('reset_memory:', mcp_server.reset_memory())

## Step 5 — how Claude Desktop registers the server

In Claude Desktop's config (`~/Library/Application Support/Claude/claude_desktop_config.json` on Mac), users add:

```jsonc
{
  "mcpServers": {
    "engine": {
      "command": "python",
      "args": ["-m", "engine.mcp.server"],
      "env": {
        "OPENAI_BASE_URL": "http://localhost:11434/v1",
        "OPENAI_API_KEY":  "ollama",
        "MODEL_SYNTHESIZER": "gemma3:4b",
        "SEARXNG_URL":    "http://localhost:8888"
      }
    }
  }
}
```

Then `/research What is X?` in Claude Desktop invokes the server's `research` tool and renders the structured JSON.

The same JSON blob is in `engine/mcp/claude_plugin/.claude-plugin/plugin.json` — that file is what gets submitted to the Anthropic plugin marketplace.

In [ ]:
# Show the plugin.json that accompanies this MCP server
!cat /content/engine-repo/engine/mcp/claude_plugin/.claude-plugin/plugin.json | python -m json.tool

## What to try next

- Spawn `python -m engine.mcp.server` as a subprocess and connect with the official `mcp` Python client — exercises the real stdio wire protocol.
- Use `npx @modelcontextprotocol/inspector python -m engine.mcp.server` to get a browser UI for invoking the tools.
- Register the server in Claude Desktop and try `/research` there.

Next tutorial: **05 — Domain presets showcase** — compare all 6 presets on the same question.